In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

# 1. Hardware Verification & Hardware Acceleration Setup
assert torch.cuda.is_available() and "A100" in torch.cuda.get_device_name(0), "Ensure you are using your rented A100 GPU!"
model_id = "google/gemma-4-E2B"

# 2. Load Tokenizer and Dataset
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token # Base models need a padding token defined

dataset = load_dataset("json", data_files={"train": "it-by-stephen-king.jsonl"})

# 3. Load the Base Model in Full bfloat16 Precision
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# 4. Define target modules and configure LoRA Adapters
peft_config = LoraConfig(
    r=64,                                # Rank size (increase to 32/64 if task is highly complex)
    lora_alpha=128,                       # Scaling factor (usually 2x Rank)
    target_modules=[
        "q_proj.linear",
        "k_proj.linear",
        "v_proj.linear",
        "o_proj.linear",
        "gate_proj.linear",
        "up_proj.linear",
        "down_proj.linear",
    ], # Targets all Gemma 4 linear layers
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()       # Verifies adapters are hooked up properly

# 5. Define Training Hyperparameters
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./gemma-4-lora-output",

    # Training
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate = 1e-4,
    lr_scheduler_type = "cosine",
    warmup_ratio = 0.03,
    weight_decay = 0.01,
    num_train_epochs=1,

    # Precision
    bf16=True,

    # Logging / Saving
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",

    # SFT-specific
    max_length=4096,
)

# 6. Initialize SFTTrainer and run training loop
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    processing_class=tokenizer,
)

print("Starting LoRA fine-tuning loop...")
trainer.train()

# 7. Save the trained adapter weights (not the whole model)
trainer.model.save_pretrained("./final_lora_adapter")
tokenizer.save_pretrained("./final_lora_adapter")
print("Training complete! Adapter weights saved.")


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 22,806,528 || all params: 5,127,104,032 || trainable%: 0.4448


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 1}.


Starting LoRA fine-tuning loop...


Step,Training Loss
10,2.859346
